In [ ]:
import torch, json
gpu_check = {'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
print(json.dumps(gpu_check, indent=2))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.rand(2, 2, device=device) @ torch.rand(2, 2, device=device))

In [ ]:
%pip -q install ultralytics roboflow
import ultralytics
print('ultralytics_version:', ultralytics.__version__)
from google.colab import userdata
try:
    rf_key = userdata.get('ROBOFLOW_API_KEY')
    print('ROBOFLOW_API_KEY available:', bool(rf_key))
except Exception as exc:
    rf_key = None
    print('ROBOFLOW_SECRET_STATUS:', type(exc).__name__)


In [ ]:
from ultralytics import YOLO
import numpy as np, pathlib, time
preflight_model = YOLO('yolo11n.pt')
blank = np.zeros((640, 640, 3), dtype=np.uint8)
preflight_result = preflight_model.predict(blank, device=0, imgsz=640, verbose=False)[0]
print('PRETRAINED_MODEL_PREFLIGHT_ONLY — not five-class training or accuracy evaluation')
print('model_device:', next(preflight_model.model.parameters()).device)
print('pretrained_classes:', len(preflight_model.names))
print('weight_bytes:', pathlib.Path('yolo11n.pt').stat().st_size)
print('blank_image_speed_ms:', preflight_result.speed)
print('blank_image_detections:', len(preflight_result.boxes))


In [ ]:
print('actual_predictor_device:', preflight_model.predictor.device)
print('actual_backend_parameter_device:', next(preflight_model.predictor.model.model.parameters()).device)
assert str(preflight_model.predictor.device).startswith('cuda')
print('YOLO11n GPU preflight passed; training/data evaluation NOT started.')


In [ ]:
from google.colab import userdata
try:
    rf_key = userdata.get('ROBOFLOW_API_KEY')
    print('ROBOFLOW_SECRET_AVAILABLE:', bool(rf_key and rf_key.strip()))
except Exception as exc:
    rf_key = None
    print('ROBOFLOW_SECRET_STATUS:', type(exc).__name__)


In [ ]:
import contextlib, io
from pathlib import Path
from roboflow import Roboflow
SOURCE_ROOT = None
print('Downloading FitFuel/all-gym-equipment v2; credential output suppressed.')
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        rf = Roboflow(api_key=rf_key)
        dataset = rf.workspace('fitfuel').project('all-gym-equipment').version(2).download('yolov8', location='/content/skku1398_workspace/raw_source/fitfuel_v2')
    SOURCE_ROOT = Path(dataset.location)
    print('DATASET_DOWNLOAD_OK:', str(SOURCE_ROOT))
    print('DATA_YAML_PRESENT:', (SOURCE_ROOT / 'data.yaml').is_file())
except Exception as exc:
    print('DATASET_DOWNLOAD_ERROR_TYPE:', type(exc).__name__)
    response = getattr(exc, 'response', None)
    print('HTTP_STATUS:', getattr(response, 'status_code', None))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML')
print('PROJECT_FOLDER_EXISTS:', DRIVE_ROOT.is_dir())


In [ ]:
import json, shutil, yaml, sys
from datetime import datetime
from pathlib import Path
WORK = Path('/content/skku1398_workspace')
original_nb = json.loads((DRIVE_ROOT/'01_notebooks/SKKU_1398_5class_Colab_Training.ipynb').read_text())
exec(''.join(original_nb['cells'][8]['source']))
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
PREPARED = WORK/'prepared'/('audited_'+RUN_TAG)
report = prepare_yolo_dataset(SOURCE_ROOT, PREPARED, mode='baseline', seed=42)
print(json.dumps(report, ensure_ascii=False, indent=2))
AUDIT_DRIVE = DRIVE_ROOT/'02_dataset'/('fitfuel_v2_'+RUN_TAG)
AUDIT_DRIVE.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREPARED/'reports', AUDIT_DRIVE/'initial_audit', dirs_exist_ok=True)
print('AUDIT_SAVED:', AUDIT_DRIVE)
print('SOURCE_IMAGE_SAMPLES:', [p.name for p in (SOURCE_ROOT/'train/images').glob('*')][:5])


In [ ]:
from collections import Counter
import skku1398_ml as ml
cfg = yaml.safe_load((SOURCE_ROOT/'data.yaml').read_text())
print('SOURCE_NAMES:', cfg['names'])
for sp in ['train','valid','test']:
    imgs = list((SOURCE_ROOT/sp/'images').glob('*'))
    labs = list((SOURCE_ROOT/sp/'labels').glob('*.txt'))
    cc = Counter(int(line.split()[0]) for p in labs for line in p.read_text().splitlines() if line.strip())
    print(sp, 'images',len(imgs),'labels',len(labs),'boxes',dict(cc))
    print('samples:', [(p.name,p.read_text()[:160]) for p in labs[:2]])
print('MAPPING:', ml.source_to_target(ml.parse_class_names(cfg['names'])))
print('TEST_ERRORS:', [ml.read_yolo_boxes(p, ml.source_to_target(ml.parse_class_names(cfg['names']))) for p in list((SOURCE_ROOT/'test/labels').glob('*.txt'))[:5]])


In [ ]:
%pip -q install ImageHash
import runpy
audit_state = runpy.run_path(str(DRIVE_ROOT/'01_notebooks/skku_audit.py'))
PREPARED = audit_state['PREP']


In [ ]:
import csv
R = DRIVE_ROOT/'02_dataset/fitfuel_v2_grouped_v1'
manifest = list(csv.DictReader((PREPARED/'reports/manifest.csv').open(encoding='utf-8-sig')))
frozen = json.loads((R/'frozen_split_manifest.json').read_text())
by_file = {str(audit_state['OUT']/r['file']):r['group'] for r in frozen}
near = list(csv.DictReader((PREPARED/'reports/potential_near_duplicates.csv').open(encoding='utf-8-sig')))
priority={'train':2,'val':1,'test':0}
exclude=set()
for pair in near:
    side='a' if priority[pair['split_a']]>priority[pair['split_b']] else 'b'
    exclude.add(by_file[pair['image_'+side]])
removed=[]
for r in manifest:
    if by_file[r['source_image']] in exclude:
        p=PREPARED/r['image']; lab=p.parent.parent/'labels'/(p.stem+'.txt')
        for f in [p,lab]:
            dest=WORK/'quarantine_v1'/f.relative_to(PREPARED);dest.parent.mkdir(parents=True,exist_ok=True);shutil.move(str(f),str(dest))
        removed.append(r)
report=json.loads((R/'audit.json').read_text())
report['additional_near_duplicate_groups_quarantined']=len(exclude)
report['additional_images_quarantined']=len(removed)
report['final_counts']={}
for sp in ['train','val','test']:
    cnt=Counter(int(l.split()[0]) for p in (PREPARED/sp/'labels').glob('*.txt') for l in p.read_text().splitlines() if l.strip())
    report['final_counts'][sp]={'images':len(list((PREPARED/sp/'images').glob('*'))),'boxes':{ml.CLASS_NAMES[c]:cnt[c] for c in range(5)}}
(R/'audit.json').write_text(json.dumps(report,indent=2))
(R/'quarantined_manifest.json').write_text(json.dumps(removed,indent=2))
(R/'final_training_manifest.json').write_text(json.dumps([r for r in manifest if r not in removed],indent=2))
shutil.copy2(shutil.make_archive(str(WORK/'five_class_grouped_v1_final'),'zip',PREPARED),R/'five_class_grouped_v1_final.zip')
print(json.dumps(report['final_counts'],indent=2));print('QUARANTINED:',len(removed))
from IPython.display import display
from PIL import Image
display(Image.open(R/'train_label_review.jpg'))


In [ ]:
import runpy
training_state = runpy.run_path(str(DRIVE_ROOT/'01_notebooks/skku_train.py'))


In [ ]:
import subprocess, sys, json
from pathlib import Path
RUN = Path((DRIVE_ROOT/'04_exports_and_reports/latest_run.txt').read_text().strip())
assert json.loads((RUN/'status.json').read_text())['evaluation']=='completed'
print('Starting LiteRT export; log:', RUN/'export.log', flush=True)
with (RUN/'export.log').open('w') as log:
    export_process = subprocess.run([sys.executable, str(DRIVE_ROOT/'01_notebooks/skku_export.py')], stdout=log, stderr=subprocess.STDOUT)
print('EXPORT_EXIT_CODE:', export_process.returncode)
print((RUN/'export.log').read_text()[-8000:])


In [ ]:
%pip -q install fastapi python-multipart httpx
import subprocess, sys
with (RUN/'final_report.log').open('w') as log:
    finish_process=subprocess.run([sys.executable,str(DRIVE_ROOT/'01_notebooks/skku_finish.py')],stdout=log,stderr=subprocess.STDOUT)
print('FINISH_EXIT',finish_process.returncode)
print((RUN/'final_report.log').read_text()[-6000:])

In [ ]:
import json, shutil, hashlib, runpy, subprocess, sys
DEST=DRIVE_ROOT/'04_exports_and_reports'/RUN.name
p=DRIVE_ROOT/'01_notebooks/skku_inference.py'
s=p.read_text().replace('confidence=.25,normalized=False):','confidence=.25,normalized=True):')
p.write_text(s); (DEST/'skku_inference.py').write_text(s)
helper=runpy.run_path(str(DEST/'skku_inference.py'))
photo=sorted((PREPARED/'test/images').glob('*'))[0]
result=helper['predict_litert'](photo,DEST/'skku1398_yolo11n_fp32.tflite')
assert result['detections'] and result['detections'][0]['bbox_xyxy_pixels'][2]>100
(DEST/'python_litert_default_test.json').write_text(json.dumps(result,indent=2))
(DEST/'requirements_executed.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
status={'smoke':'passed','baseline':'completed_30_epochs_T4','public_test':'completed','litert_export':'passed','python_inference':'passed','http_api':'passed_200','center_validation':False,'flutter_phone_test':False,'run':RUN.name}
for target in [RUN/'status.json',DEST/'status.json',DRIVE_ROOT/'04_exports_and_reports/environment_status.json']:
    target.write_text(json.dumps(status,indent=2))
with (DEST/'REPORT_KO.md').open('a') as f:
    f.write('\n실행 환경 참고: API 패키지 설치 시 xdsl 0.28.0과 typing-extensions 4.16.0의 의존성 경고가 발생했습니다. 기존 LiteRT 변환과 이후 interpreter/API 실제 호출은 성공했습니다. requirements_executed.txt에 설치 버전을 기록했습니다.\n')
integrity={str(p.relative_to(DEST)):{'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in DEST.rglob('*') if p.is_file() and p.name!='artifact_integrity.json'}
(DEST/'artifact_integrity.json').write_text(json.dumps(integrity,indent=2))
print('FINAL_VERIFIED',json.dumps(status))
print('BENCHMARK',(RUN/'inference_benchmark.json').read_text())
print('MODEL_BYTES',integrity['skku1398_yolo11n_fp32.tflite']['bytes'],integrity['skku1398_yolo11n_best.pt']['bytes'])
print('DELIVERABLE_FILES',len(integrity))
from google.colab import drive
drive.flush_and_unmount()
print('DRIVE_SYNC_COMPLETE')

In [ ]:
%pip -q install roboflow pyyaml
import contextlib, io, json, re
from pathlib import Path
from collections import Counter, defaultdict
from google.colab import userdata
from roboflow import Roboflow
import yaml
specs=[('fitfuel','all-gym-equipment',2),('rizzlabzz','gymbro',2),('trainers-workspace','gym-equipment-t6kck-fkuoc',1),('gymze','gymze',1)]
counts_report={}
for workspace, project, version in specs:
    print('START',workspace,flush=True)
    try:
        root=Path('/content/equipment_counts')/workspace
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            if not (root/'data.yaml').exists():
                ds=Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(workspace).project(project).version(version).download('yolov8',location=str(root))
        names=yaml.safe_load((root/'data.yaml').read_text())['names']
        names=dict(enumerate(names)) if isinstance(names,list) else names
        boxes=Counter(); images=Counter(); groups=defaultdict(set); splits={}
        for split in ['train','valid','test']:
            sc=Counter()
            for f in (root/split/'labels').glob('*.txt'):
                ids=[int(x.split()[0]) for x in f.read_text().splitlines() if x.strip()]
                boxes.update(ids); images.update(set(ids)); sc.update(set(ids))
                stem=re.sub(r'[-_]mirrored','',f.stem.split('.rf.')[0],flags=re.I)
                for c in set(ids): groups[c].add(stem)
            splits[split]={names[c]:n for c,n in sc.items()}
        rows={names[c]:{'images':images[c],'boxes':boxes[c],'filename_groups_not_deduplicated':len(groups[c])} for c in sorted(names)}
        counts_report[workspace]={'classes':rows,'split_images':splits}
        print('RESULT',workspace,json.dumps(rows),flush=True)
        Path('/content/equipment_counts_report.json').write_text(json.dumps(counts_report,indent=2))
    except Exception as exc:
        print('FAILED',workspace,type(exc).__name__,flush=True)
print('COUNT_COMPLETE',flush=True)


In [ ]:
specs=[('gym-qknei','gym-shkud',5),('dans-new-workspace','home-gym-equipment-detection',6),('hn-2kixi','healnion-f4l32',1)]
exec('for workspace'+In[20].split('for workspace',1)[1])


In [ ]:
import math
for workspace in counts_report:
    root=Path('/content/equipment_counts')/workspace
    missing=bad=labels=0
    for split in ['train','valid','test']:
        stems={p.stem for p in (root/split/'images').glob('*') if p.suffix.lower() in ['.jpg','.jpeg','.png','.webp','.bmp']}
        for f in (root/split/'labels').glob('*.txt'):
            labels+=1; missing+=int(f.stem not in stems)
            for line in f.read_text().splitlines():
                if not line.strip(): continue
                v=list(map(float,line.split()))
                bad+=int(len(v)!=5 or not all(math.isfinite(x) for x in v) or any(x<0 or x>1 for x in v[1:]) or v[3]<=0 or v[4]<=0)
    print(workspace,'label_files=',labels,'missing_images=',missing,'invalid_box_rows=',bad)
print('7 DATASETS COUNTED; AUGMENTATIONS INCLUDED; CENTER PERFORMANCE NOT VERIFIED')


In [ ]:
valid_report={}
for workspace in counts_report:
    root=Path('/content/equipment_counts')/workspace
    names=yaml.safe_load((root/'data.yaml').read_text())['names']
    names=dict(enumerate(names)) if isinstance(names,list) else names
    imgs=Counter(); boxes=Counter(); groups=defaultdict(set)
    for split in ['train','valid','test']:
        for f in (root/split/'labels').glob('*.txt'):
            ids=[]
            for line in f.read_text().splitlines():
                if not line.strip(): continue
                v=list(map(float,line.split()))
                if len(v)!=5 or not all(math.isfinite(x) for x in v) or any(x<0 or x>1 for x in v[1:]) or v[3]<=0 or v[4]<=0: continue
                ids.append(int(v[0]))
            imgs.update(set(ids));boxes.update(ids)
            for c in set(ids):groups[c].add(re.sub(r'[-_]mirrored','',f.stem.split('.rf.')[0],flags=re.I))
    valid_report[workspace]={names[c]:[imgs[c],boxes[c],len(groups[c])] for c in names}
    print('VALID',workspace,json.dumps(valid_report[workspace]))
print('BASIC_VALIDATION_COMPLETE; counts are images, boxes, filename groups; not visually audited')


In [ ]:
import contextlib,io,json,re,math,hashlib
from pathlib import Path
from collections import Counter,defaultdict
from google.colab import userdata
from roboflow import Roboflow
import yaml
new_specs=[('hydraws','exercise-identifier',2),('shazils-workspace','gym-0vnct-2p6bi',1),('xuejun-hong','gym-equipment-iv6k9',1)]
new_reports={}; new_records={}
for ws,proj,ver in new_specs:
    print('DOWNLOAD_START',ws,flush=True)
    try:
        root=Path('/content/equipment_search2')/ws
        with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
            if not (root/'data.yaml').exists():
                Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(ws).project(proj).version(ver).download('yolov8',location=str(root))
        names=yaml.safe_load((root/'data.yaml').read_text())['names']
        names=dict(enumerate(names)) if isinstance(names,list) else names
        rec=[]; bad=missing=0; polygons=0
        for sp in ['train','valid','test']:
            imap={p.stem:p for p in (root/sp/'images').glob('*') if p.is_file()}
            for f in (root/sp/'labels').glob('*.txt'):
                if f.stem not in imap: missing+=1;continue
                boxes=[]
                for line in f.read_text().splitlines():
                    if not line.strip():continue
                    v=list(map(float,line.split())); coords=v[1:]
                    if not all(math.isfinite(x) for x in v) or any(x<0 or x>1 for x in coords) or int(v[0]) not in names:bad+=1;continue
                    if len(v)==5: cx,cy,w,h=coords
                    elif len(coords)>=6 and len(coords)%2==0:
                        xs=coords[::2];ys=coords[1::2];cx=(min(xs)+max(xs))/2;cy=(min(ys)+max(ys))/2;w=max(xs)-min(xs);h=max(ys)-min(ys);polygons+=1
                    else:bad+=1;continue
                    if w<=0 or h<=0:bad+=1;continue
                    boxes.append([int(v[0]),cx,cy,w,h])
                if boxes:rec.append({'image':str(imap[f.stem]),'stem':f.stem,'split':sp,'boxes':boxes})
        new_records[ws]=rec
        cc=Counter();bc=Counter();groups=defaultdict(set)
        for r in rec:
            ids=[b[0] for b in r['boxes']];cc.update(set(ids));bc.update(ids)
            for c in set(ids):groups[c].add(r['stem'].split('.rf.')[0])
        rows={names[c]:{'images':cc[c],'boxes':bc[c],'filename_groups':len(groups[c])} for c in names}
        new_reports[ws]={'source':f'https://universe.roboflow.com/{ws}/{proj}/dataset/{ver}','classes':rows,'bad_rows':bad,'missing_images':missing,'polygon_rows_converted':polygons}
        print('NEW_RESULT',ws,json.dumps(new_reports[ws]),flush=True)
    except Exception as e:print('FAILED',ws,type(e).__name__,flush=True)
print('NEW_COUNTS_DONE',flush=True)


In [ ]:
from ultralytics import YOLO
from google.colab import drive
from google.colab.patches import cv2_imshow
from pathlib import Path

# 1. Google Drive 다시 마운트 (이전 셀에서 unmount 되었으므로 필요)
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Drive mount failed:", e)

# 2. 학습된 PyTorch 모델 불러오기
model_path = '/content/drive/MyDrive/SKKU_1398_Fitness_ML/04_exports_and_reports/grouped_v1_20261026T075807Z/skku1398_yolo11n_best.pt'
model = YOLO(model_path)

# 3. 로컬 워크스페이스에 준비된 실제 테스트 이미지 탐색
test_images_dir = Path('/content/skku1398_workspace/prepared/grouped_v1/test/images')
if not test_images_dir.exists():
    # 대안 경로 탐색
    test_images_dir = Path('/content/skku1398_workspace/raw_source/fitfuel_v2/test/images')

test_images = list(test_images_dir.glob('*'))

if test_images:
    image_path = str(test_images[0])
    print(f"Testing with existing image: {image_path}")

    # 4. 추론 실행
    results = model(image_path)

    # 5. 결과 시각화 및 화면 출력
    for r in results:
        im_bgr = r.plot()  # 바운딩 박스가 그려진 이미지(BGR numpy array)
        cv2_imshow(im_bgr)
else:
    print("테스트용 이미지를 찾을 수 없습니다. 경로를 확인해 주세요.")

In [ ]:
from pathlib import Path
import json
print('BUILD_PREFLIGHT', json.dumps({p: Path(p).exists() for p in ['/content/equipment_counts','/content/equipment_search2','/content/drive/MyDrive/SKKU_1398_Fitness_ML']}))
from google.colab import userdata
try:
    print('ROBOFLOW_SECRET_AVAILABLE',bool(userdata.get('ROBOFLOW_API_KEY')))
except Exception as e:
    print('ROBOFLOW_SECRET_STATUS',type(e).__name__)


In [ ]:
import subprocess, sys
from pathlib import Path
script=Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/01_notebooks/build_equipment_v1.py')
assert script.exists(), 'Wait for Drive synchronization'
build_log=Path('/content/equipment_build_v1.log')
with build_log.open('w') as log:
    build_proc=subprocess.Popen([sys.executable,str(script)],stdout=log,stderr=subprocess.STDOUT)
print('BUILD_STARTED_PID',build_proc.pid)
print('SCRIPT_BYTES',script.stat().st_size)


In [ ]:
print('PROCESS_EXIT',build_proc.poll())
print(build_log.read_text()[-7000:])


In [ ]:
import psutil, hashlib
print('ACTIVE_BUILD',[(p.pid, p.create_time()) for p in psutil.process_iter() if 'build_equipment_v1.py' in ' '.join(p.cmdline())])
print('SCRIPT_BYTES_NOW',script.stat().st_size,'SHA256',hashlib.sha256(script.read_bytes()).hexdigest())
print('BUILD_LOG',build_log.read_text()[-5000:])
print('RECENT_BUILD_ACTIONS',[(i, 'build_equipment_v1' in In[i], 'terminate' in In[i], 'write_text' in In[i]) for i in range(38,min(42,len(In)))])
import psutil, subprocess, sys, time
from pathlib import Path
script=Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/01_notebooks/build_equipment_v1.py')
assert 'RESUMED_INDEX' in script.read_text()
prior=[p for p in psutil.process_iter() if str(script) in p.cmdline()]
for p in prior: p.terminate()
psutil.wait_procs(prior,timeout=5)
build_log=Path('/content/equipment_build_v1_resume.log')
with build_log.open('w') as log:
    build_proc=subprocess.Popen([sys.executable,str(script)],stdout=log,stderr=subprocess.STDOUT)
print('SINGLE_RESUMABLE_BUILD',build_proc.pid,'prior_processes',len(prior))


In [ ]:
from pathlib import Path
work=Path('/content/SKKU1398_equipment_v1_work')
print('PROCESS_EXIT',build_proc.poll())
print('REVIEW_IMAGES',len(list((work/'review_pack/images').glob('*'))))
print('CONTACT_SHEETS',len(list((work/'review_pack/contact_sheets').glob('*.jpg'))))
print('LATEST_LOG',build_log.read_text()[-1800:])


In [ ]:
import subprocess,sys
assert build_proc.wait(timeout=45)==0,build_log.read_text()[-2000:]
result=subprocess.run([sys.executable,'/content/drive/MyDrive/SKKU_1398_Fitness_ML/01_notebooks/package_review_material.py'],capture_output=True,text=True)
print(result.stdout)
assert result.returncode==0,result.stderr
print('AUTOMATED_BUILD_VERIFIED',Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/02_dataset/SKKU1398_equipment_v1/reports/automated_build_summary.json').read_text())


In [ ]:
from pathlib import Path
import subprocess,sys,json
root_v1=Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/02_dataset/SKKU1398_equipment_v1')
assert (root_v1/'review/visual_decisions.json').exists(), 'Drive decisions not synced yet'
export_result=subprocess.run([sys.executable,'/content/drive/MyDrive/SKKU_1398_Fitness_ML/01_notebooks/export_reviewed_v1.py'],capture_output=True,text=True)
print(export_result.stdout)
assert export_result.returncode==0,export_result.stderr
for partial in (root_v1/'raw_sources').glob('*.incomplete_*'):
    partial.rename(root_v1/'reports'/partial.name)
print('REVIEWED_YOLO_EXPORT_VERIFIED')

In [ ]:
from IPython.display import display, Markdown
s=json.loads((root_v1/'reports/reviewed_export_summary.json').read_text())
a=json.loads((root_v1/'reports/automated_build_summary.json').read_text())
display(Markdown(f"## SKKU1398 equipment v1 — 파일럿 구축 완료\n\n원본 **{a['raw_sources']}개 출처 / {a['raw_images']:,}장** 보존 · 검수 대표 **{a['review_representative_images']:,}장**\n\n우선 4종 **{s['reviewed_images']:,}장 시각 선별 → {s['exported_images']}장 승인**\n\n리컴번트 86 · 일립티컬 44 · 케이블 크로스오버 38 · 하이퍼익스텐션 35\n\n학습 163 / 검증 22 / 테스트 18 · 알려진 중복 그룹 분할 교차 0\n\n**YOLO ZIP 저장·무결성 검사 완료. 모델 학습 및 센터 실물 검증은 미실시.**"))